### 🛒 Retail Demand Forecasting

#### Phase 1: Data Understanding

##### 1. Objective

The objective of this notebook is to understand the available datasets, identify their structure, analyze data quality, and understand how the datasets are related before performing exploratory data analysis and feature engineering.# New Section

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)


#### Load Data

In [56]:
train = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/train.csv")
test = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/test.csv")
items = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/items.csv")
stores = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/stores.csv")
oil = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/oil.csv")
transactions = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/transactions.csv")
holidays = pd.read_csv("/content/drive/MyDrive/Retail-Demand-Forecasting/Data/Raw/holidays_events.csv")

#### Dataset summary

In [33]:
datasets = {"train":train,
            "test" : test,
            "items" :items,
            "stores" : stores,
            "oil" : oil,
            "transactions" : transactions,
            "holidays" :holidays,
            }

summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
    "Missing Values": [df.isnull().sum().sum() for df in datasets.values()],
    "Missing %": [round((df.isnull().sum().sum() / df.size) * 100, 2) for df in datasets.values()],
    "Memory Usage (MB)": [round(df.memory_usage(deep=True).sum() / (1024**2), 2) for df in datasets.values()]
})

summary


,Dataset,Rows,Columns,Missing Values,Missing %,Memory Usage (MB)
0,train,125497040,6,21657651,2.88,15117.16
1,test,3370464,5,0,0.00,270.00
2,items,4100,4,0,0.00,0.32
3,stores,54,5,0,0.00,0.01
4,oil,1218,2,43,1.77,0.08
5,transactions,83488,3,0,0.00,5.97
6,holidays,350,6,0,0.00,0.10


In [34]:
summary

,Dataset,Rows,Columns,Missing Values,Missing %,Memory Usage (MB)
0,train,125497040,6,21657651,2.88,15117.16
1,test,3370464,5,0,0.00,270.00
2,items,4100,4,0,0.00,0.32
3,stores,54,5,0,0.00,0.01
4,oil,1218,2,43,1.77,0.08
5,transactions,83488,3,0,0.00,5.97
6,holidays,350,6,0,0.00,0.10


### Key Observations
* The train dataset is the largest dataset, containing over 125 million records, indicating a large-scale retail demand forecasting problem.
Approximately 2.88% of the values are missing in the train dataset.
* Further investigation is required to identify the affected columns before deciding on an appropriate treatment strategy.
* The oil dataset contains 43 missing values (1.77%). Due to its relatively small size, dropping records may not be the best approach, and suitable imputation methods should be evaluated.
* All remaining datasets are complete and do not contain missing values.
* The train dataset consumes approximately 15 GB of memory, suggesting that memory-efficient processing techniques may be required during model development.

In [57]:
from IPython.display import display

def observations(df):
    print("=" * 80)
    print("First 5 Rows")
    display(df.head())

    print("=" * 80)
    print("Shape")
    display(df.shape)

    print("=" * 80)
    print("Data Information")
    df.info()

    print("=" * 80)
    print("Descriptive Statistics")
    display(df.describe(include="all"))

    print("=" * 80)
    print("Unique Values")
    display(df.nunique())

    print("=" * 80)
    print("Data Types")
    display(df.dtypes)

    print("=" * 80)
    print("Missing Values")
    display(df.isnull().sum())

    return df

### Train Dataset Analysis

In [50]:
observations(train)

First 5 Rows


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


Shape


(125497040, 6)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   item_nbr     int64  
 4   unit_sales   float64
 5   onpromotion  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 5.6+ GB
Descriptive Statistics


...

Data Types


,0
id,int64
date,object
store_nbr,int64
item_nbr,int64
unit_sales,float64
onpromotion,object


Missing Values


,0
id,0
date,0
store_nbr,0
item_nbr,0
unit_sales,0
onpromotion,21657651


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


#### Key Observations

- The train dataset contains **125.5 million records**, making it the largest dataset in the project.
- The dataset contains **6 columns**, including the target variable (`unit_sales`) and promotion information (`onpromotion`).
- The `date` column is currently stored as an **object** and should be converted to **datetime** for time-series analysis.
- There are **54 unique stores** and **4,036 unique products**, indicating forecasting at the Store–Item level.
- The `onpromotion` column contains approximately **17.26% missing values**, while all other columns are complete.
- The dataset occupies approximately **5.6 GB** of memory, which may require memory-efficient processing during feature engineering and model training.

#### Business Interpretation

- Historical sales data will be used to train forecasting models.
- Promotion is expected to be one of the most important drivers of product demand.
- Since only the promotion column contains missing values, data quality issues appear to be limited.
- The large dataset size suggests that scalable preprocessing and modeling techniques will be important.

### Test Dataset Analysis

In [58]:
observations(test)


First 5 Rows


,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,False
1,125497041,2017-08-16,1,99197,False
2,125497042,2017-08-16,1,103501,False
3,125497043,2017-08-16,1,103520,False
4,125497044,2017-08-16,1,103665,False


Shape


(3370464, 5)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3370464 entries, 0 to 3370463
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   id           int64 
 1   date         object
 2   store_nbr    int64 
 3   item_nbr     int64 
 4   onpromotion  bool  
dtypes: bool(1), int64(3), object(1)
memory usage: 106.1+ MB
Descriptive Statistics


...

Data Types


,0
id,int64
date,object
store_nbr,int64
item_nbr,int64
onpromotion,bool


Missing Values


,0
id,0
date,0
store_nbr,0
item_nbr,0
onpromotion,0


,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,False
1,125497041,2017-08-16,1,99197,False
2,125497042,2017-08-16,1,103501,False
3,125497043,2017-08-16,1,103520,False
4,125497044,2017-08-16,1,103665,False
...,...,...,...,...,...
3370459,128867499,2017-08-31,54,2132163,False
3370460,128867500,2017-08-31,54,2132318,False
3370461,128867501,2017-08-31,54,2132945,False
3370462,128867502,2017-08-31,54,2132957,False


### key Observations

* The test dataset contains 3,901 products compared to 4,036 products in the training dataset.

* This suggests that some products are absent from the forecasting period. Further investigation is required to determine whether these products were discontinued, seasonal, or intentionally excluded from the prediction dataset.

* The onpromotion column is stored as a boolean variable in the test dataset, whereas it is an object in the training dataset due to the presence of missing values in the training data.

In [59]:
observations(oil)

First 5 Rows


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


Shape


(1218, 2)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB
Descriptive Statistics


...

Data Types


,0
date,object
dcoilwtico,float64


Missing Values


,0
date,0
dcoilwtico,43


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
...,...,...
1213,2017-08-25,47.65
1214,2017-08-28,46.40
1215,2017-08-29,46.46
1216,2017-08-30,45.96


In [68]:
store = observations(stores)


First 5 Rows


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


Shape


(54, 5)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB
Descriptive Statistics


...

,0
store_nbr,54
city,22
state,16
type,5
cluster,17


Data Types


,0
store_nbr,int64
city,object
state,object
type,object
cluster,int64


Missing Values


,0
store_nbr,0
city,0
state,0
type,0
cluster,0


### Key Observations (stores)

- The dataset contains metadata for all 54 stores used in the project.
- Stores are distributed across 22 cities and 16 states, providing geographical information.
- Stores are categorized into 5 different types and 17 predefined clusters.
- No missing values are present in the dataset.
- The dataset can be merged with the sales data using `store_nbr` to enrich the forecasting model with store-level features.

### Business Interpretation

- Geographic attributes such as city and state can capture regional demand variations.
- Store type and cluster may explain differences in customer behavior and store characteristics.
- These attributes are expected to improve forecasting by incorporating store-specific information beyond historical sales.

In [70]:
observations(transactions)


First 5 Rows


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


Shape


(83488, 3)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB
Descriptive Statistics


...

Data Types


,0
date,object
store_nbr,int64
transactions,int64


Missing Values


,0
date,0
store_nbr,0
transactions,0


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


### Business Interpretation (Transactions)

- The transactions dataset captures daily customer activity at each store.
- Customer traffic is expected to have a strong relationship with sales volume.
- Transaction counts can be used as an external feature to improve forecasting performance.
- The dataset can be merged with the sales data using both `date` and `store_nbr`.

In [65]:
observations(items)


First 5 Rows


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


Shape


(4100, 4)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB
Descriptive Statistics


...

Data Types


,0
item_nbr,int64
family,object
class,int64
perishable,int64


Missing Values


,0
item_nbr,0
family,0
class,0
perishable,0


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1
...,...,...,...,...
4095,2132318,GROCERY I,1002,0
4096,2132945,GROCERY I,1026,0
4097,2132957,GROCERY I,1068,0
4098,2134058,BEVERAGES,1124,0


### Business Interpretation (Items)

- Product family captures broad product categories and can help identify category-specific demand patterns.
- Product class provides a more granular categorization within each family.
- The perishable flag is expected to be an important predictor, as demand patterns for perishable products differ significantly from non-perishable products.
- Product attributes can be merged with the sales dataset using `item_nbr` to enrich the forecasting model.

In [67]:
holiday = observations(holidays)
holiday

First 5 Rows


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


Shape


(350, 6)

Data Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB
Descriptive Statistics


...

Data Types


,0
date,object
type,object
locale,object
locale_name,object
description,object
transferred,bool


Missing Values


,0
date,0
type,0
locale,0
locale_name,0
description,0
transferred,0


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
...,...,...,...,...,...,...
345,2017-12-22,Additional,National,Ecuador,Navidad-3,False
346,2017-12-23,Additional,National,Ecuador,Navidad-2,False
347,2017-12-24,Additional,National,Ecuador,Navidad-1,False
348,2017-12-25,Holiday,National,Ecuador,Navidad,False


### Business Interpretation (Holiday)

- Holidays are expected to have a significant impact on retail demand.
- National holidays affect all stores, while regional and local holidays influence only specific geographical locations.
- Different event types (Holiday, Bridge, Additional, etc.) may have varying effects on customer purchasing behavior.
- The holiday dataset can be merged with the sales data using the date field and further filtered using store location information.

# Overall Dataset Understanding

The Corporación Favorita dataset consists of one primary sales dataset and multiple supporting datasets that provide additional business context for demand forecasting.

The **train** dataset contains historical sales recorded at the **Store × Item × Date** level and serves as the primary dataset for model training. The **test** dataset contains the future dates for which sales predictions need to be generated.

The remaining datasets provide valuable external information that can improve forecasting performance:

- **Items** dataset provides product attributes such as family, class, and perishability.
- **Stores** dataset provides geographical and operational information including city, state, store type, and cluster.
- **Transactions** dataset captures daily customer traffic at the store level.
- **Oil** dataset provides daily crude oil prices, which may reflect macroeconomic conditions in Ecuador.
- **Holidays** dataset contains national, regional, and local holidays that may influence customer purchasing behavior.
- **Sample Submission** dataset defines the expected output format for Kaggle submissions.

## Key Learnings

From the dataset understanding phase, the following hypotheses have been identified:

- Promotional activities are likely to influence product demand.
- Holidays may create temporary spikes or dips in sales.
- Customer transactions may serve as a strong indicator of sales volume.
- Product characteristics such as family and perishability may affect demand patterns.
- Store location, type, and cluster may explain regional variations in sales.
- Oil prices may indirectly impact retail demand by reflecting broader economic conditions in Ecuador.

## Data Quality Summary

- The train dataset contains missing values only in the `onpromotion` column.
- The oil dataset contains a small number of missing oil prices.
- All other datasets are complete and contain no missing values.
- Date columns will be converted to datetime format before analysis.

## Next Steps

The next phase of the project will focus on Exploratory Data Analysis (EDA), where these hypotheses will be validated by analyzing:

- Sales trends over time
- Seasonality and cyclic patterns
- Promotion impact on sales
- Holiday impact on demand
- Product family performance
- Store-wise and regional sales patterns
- Relationship between oil prices and sales
- Relationship between customer transactions and sales

The insights obtained from EDA will guide feature engineering and model selection for the forecasting pipeline.